## Create a Custom State for your Stategraph
Adding additional information to the state makes it easily accessible by other graph nodes (like a downstream node that stores or processes the information), as well as the graph's persistence layer.

In [24]:
# Start by loading environment variables (OpenAI model endpoint and key)
import os

from dotenv import load_dotenv
load_dotenv('.env')
OPENAI_API_ENDPOINT=os.getenv("OPENAI_API_ENDPOINT")
OPENAI_API_KEY=os.getenv("OPENAI_API_KEY")
AZURE_OPENAI_DEPLOYMENT_NAME=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME")
OPENAI_API_VERSION=os.getenv("OPENAI_API_VERSION")
TAVILY_API_KEY=os.getenv("TAVILY_API_KEY")

In [25]:
# Import libraries to build custom state:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END

from langchain_core.messages import ToolMessage
from langchain_tavily import TavilySearch
from langchain_core.tools import InjectedToolCallId, tool
from langgraph.types import Command, interrupt
from langchain.chat_models import init_chat_model
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import InMemorySaver

In [26]:
# Define the LLM to use in the graph:
llm = init_chat_model(
    model="azure_openai:gpt-5-mini",
    model_provider='azure_openai',
    azure_endpoint=OPENAI_API_ENDPOINT,
    api_key=OPENAI_API_KEY,
    azure_deployment=AZURE_OPENAI_DEPLOYMENT_NAME,
    api_version=OPENAI_API_VERSION
)

In [27]:
# Define a custom state representing a person with name and birthday:
class PersonState(TypedDict):
    messages: Annotated[list, add_messages]
    name: str
    birthday: str

In [28]:
# Create a tool that prompts a human to verify the information:
@tool
def human_assistance(
    name: str, birthday: str, tool_call_id: Annotated[str, InjectedToolCallId]
) -> str:
    """Request assistance from a human."""
    human_response = interrupt(
        {
            "question": "Is this correct?",
            "name": name,
            "birthday": birthday,
        },
    )
    # If the information is correct, update the state as-is.
    if human_response.get("correct", "").lower().startswith("y"):
        verified_name = name
        verified_birthday = birthday
        response = "Correct"
    # Otherwise, receive information from the human reviewer.
    else:
        verified_name = human_response.get("name", name)
        verified_birthday = human_response.get("birthday", birthday)
        response = f"Made a correction: {human_response}"

    # This time we explicitly update the state with a ToolMessage inside the tool
    state_update = {
        "name": verified_name,
        "birthday": verified_birthday,
        "messages": [ToolMessage(response, tool_call_id=tool_call_id)],
    }
    # We return a Command object in the tool to update our state.
    return Command(update=state_update)

In [29]:
# Build the chatbot:
tool = TavilySearch(max_results=2)
tools = [tool, human_assistance]
llm_with_tools = llm.bind_tools(tools)
graph_builder = StateGraph(PersonState)

def chatbot(state: PersonState):
    message = llm_with_tools.invoke(state["messages"])
    # Because we will be interrupting during tool execution we disable parallel tool calling to avoid
    # repeating any tool invocations when we resume
    assert len(message.tool_calls) <= 1
    return {"messages": [message]}

graph_builder.add_node("chatbot", chatbot)
tool_node = ToolNode(tools=tools)
graph_builder.add_node("tools", tool_node)

graph_builder.add_conditional_edges("chatbot", tools_condition)
graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge(START, "chatbot")

In [30]:
# Compile the graph:
memory = InMemorySaver()
graph = graph_builder.compile(checkpointer=memory)

In [31]:
# Prompt the chatbot to look up the "birthday" of the LangGraph library and direct the chatbot to reach
# out to the human_assistance tool once it has the required information.
user_input = (
    "Can you look up when LangGraph was released? "
    "When you have the answer, use the human_assistance tool for review."
)
config = {"configurable": {"thread_id": "1"}}

events = graph.stream(
    {"messages": [{"role": "user", "content": user_input}]},
    config,
    stream_mode="values",
)
for event in events:
    if "messages" in event:
        event["messages"][-1].pretty_print()

================================ Human Message =================================

Can you look up when LangGraph was released? When you have the answer, use the human_assistance tool for review.
================================== Ai Message ==================================
Tool Calls:
  tavily_search (call_TEtsuGO0enOEaSLFAC3HTtiy)
 Call ID: call_TEtsuGO0enOEaSLFAC3HTtiy
  Args:
    query: LangGraph release date LangGraph released when
    search_depth: advanced
    topic: general
================================= Tool Message =================================
Name: tavily_search

{"query": "LangGraph release date LangGraph released when", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://blog.langchain.com/langgraph-studio-the-first-agent-ide/", "title": "LangGraph Studio: The first agent IDE - LangChain Blog", "content": "In January 2023, we launched LangGraph, a highly controllable, low-level orchestration framework for building agentic applic

In [32]:
# The chatbot failed to identify the correct date, so supply it with information:
human_command = Command(
    resume={
        "name": "LangGraph",
        "birthday": "Jan 17, 2024",
    },
)

events = graph.stream(human_command, config, stream_mode="values")
for event in events:
    if "messages" in event:
        event["messages"][-1].pretty_print()

================================== Ai Message ==================================
Tool Calls:
  human_assistance (call_IAhNc9EyA3U1VBMJYMT1tipL)
 Call ID: call_IAhNc9EyA3U1VBMJYMT1tipL
  Args:
    name: Jane Doe
    birthday: 1990-01-01
================================= Tool Message =================================
Name: human_assistance

Made a correction: {'name': 'LangGraph', 'birthday': 'Jan 17, 2024'}
================================== Ai Message ==================================
Tool Calls:
  tavily_search (call_Fxxvtuv69uQEhii6xvhvsmVs)
 Call ID: call_Fxxvtuv69uQEhii6xvhvsmVs
  Args:
    query: langgraph github initial release LangGraph January 2023 LangGraph repo langchain-ai/langgraph creation date
    search_depth: advanced
    topic: general
================================= Tool Message =================================
Name: tavily_search

{"query": "langgraph github initial release LangGraph January 2023 LangGraph repo langchain-ai/langgraph creation date", "follow_up_qu

In [35]:
# Let's review the state:
snapshot = graph.get_state(config)
{k: v for k, v in snapshot.values.items() if k in ("name", "birthday")}

{'name': 'LangGraph', 'birthday': 'Jan 17, 2024'}